In [ ]:
#|default_exp cli

In [ ]:
#|export
from functools import partial

from execnb.nbio import read_nb
from fastcore.script import call_parse
from fastcore.xtras import Path, globtastic
from nbdev import nbdev_export
from nbdev.export import ExportModuleProc, nb_export
from nbdev.maker import ModuleMaker
from nbdev.processors import NBProcessor


In [ ]:
nb_path = "../nbs/losses.pytorch.ipynb"
nb = read_nb(nb_path)

In [ ]:
nb.cells

In [ ]:
#|export
tst_flags = 'datasets distributed matplotlib polars pyarrow scipy'.split()
to_skip = [
    'showdoc',
    'load_ext',
    'from nbdev'
]


def print_execs(cell):
    if 'exec' in cell.source: print(cell.source)

def print_hide(cell):
    if 'hide' in cell.directives_: print(cell.source)

def other_tests(cell):
    if len(cell.directives_) == 0:
        print(cell.source)

def get_markdown(cell):
    if cell.cell_type == "markdown":
        print(cell.source)

def get_code(cell):
    if cell.cell_type == "code":
        print(cell.source)

def extract_dir(cell, dirs):
    for directive in dirs.split(','):
        if directive in cell.directives_:
            print(cell.source)

def no_dir_and_dir(cell, dir):
    if len(cell.directives_) == 0:
        print(cell.source)

    if dir in cell.directives_:
        print(cell.source)

def get_all_tests2(cell):
    if cell.cell_type == "code":

        if len(cell.directives_) == 0:
            print(cell.source)


        elif any(x in tst_flags + ['hide'] for x in cell.directives_):
            if not (x in cell.source for x in to_skip):
                print(cell.source)

def get_all_tests(cell):
    if len(cell.directives_) == 0:
        print(cell.source)

    if any(x in tst_flags + ["hide"] for x in cell.directives_):
        print(cell.source)



In [ ]:
tst_cell = nb.cells[0]

In [ ]:
tst_cell

In [ ]:
tst_flags + ['hide']

In [ ]:
#|export
mapper = {
    'print_execs': print_execs,
    'print_hide': print_hide,
    'other_tests': other_tests,
    'get_markdown': get_markdown,
    'extract_dir': extract_dir,
    'no_dir_and_dir': no_dir_and_dir,
    'get_all_tests':get_all_tests
}

In [ ]:
#|export
@call_parse
def print_dir_in_nb(nb_path:str,
                    dir:str=None,
                    dir_name:str=None,
                    ):
    if dir_name not in mapper.keys():
        raise ValueError(f'Choose processor from the the following: {mapper.keys()}')

    if dir_name == 'extract_dir':
        processor = NBProcessor(nb_path, partial(extract_dir, dir=dir))
        processor.process()
        return
    elif dir_name == 'no_dir_and_dir':
        processor = NBProcessor(nb_path, partial(no_dir_and_dir, dir=dir))
        processor.process()
        return

    processor = NBProcessor(nb_path, mapper[dir_name])
    processor.process()


In [ ]:
NBProcessor(nb_path, procs=partial(extract_dir, dirs='hide,polars')).process()

In [ ]:
NBProcessor(nb_path, procs=get_all_tests2).process()

In [ ]:
processor = NBProcessor(nb_path, partial(extract_dir, dir='distributed'))

In [ ]:
processor.process()

In [ ]:
nb_export(nb_path, '../tests', partial(extract_dir, dir='distributed'), 'foo2')

In [ ]:
nbs = globtastic('../nbs', file_glob='*.ipynb', recursive=False).map(Path).sorted()
tst_flags = 'datasets distributed matplotlib polars pyarrow scipy'.split()

In [ ]:
nbs

In [ ]:
TEST_PATH = Path('../tests').resolve()

In [ ]:
TEST_PATH

In [ ]:
if not TEST_PATH.exists():
    TEST_PATH.mkdir()

In [ ]:
nbs[4]

In [ ]:
# for tst in tst_flags:
#     for nb in nbs:
#         nb_name = nb.stem
#         nb_export(nb, lib_path='../tests', procs=partial(extract_dir, dir=tst), name=f'{tst}_{nb_name}')

In [ ]:
# for tst in tst_flags:
#     for nb in nbs:
#         nb_name = nb.stem
#         code = NBProcessor(nb, partial(extract_dir, dir=tst))
#         with open(TEST_PATH / f'{tst}.py', '+a') as f:
#             if code.process() is not None:
#                 f.write(code.process())

In [ ]:
nb_export('cli.ipynb', lib_path='.', name='cli')